# Tutorial: Build Qdrant Index + Query Tour Timestamps (Gemini)

This notebook walks through the full pipeline in `data/`:
- ingest website pages into Qdrant using Gemini text embeddings
- ingest YouTube frames with timestamps using Gemini image embeddings
- query the index and get timestamp deeplinks for demo/tour questions

This is a **hands-on setup + runbook** for local development and testing.


## What You Will Run

1. Validate project paths and required files.
2. Install dependencies (optional cell).
3. Validate environment variables (`GOOGLE_API_KEY`, `QDRANT_URL`, etc).
4. Run ingestion script:
   - `--mode websites`
   - `--mode youtube`
   - or `--mode all`
5. Run query script and inspect top results with timestamps.

Expected outcome:
- Qdrant collection contains website chunks + YouTube frame points.
- Queries like `show me the tour demo` return YouTube timestamp links.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data'
BUILD_SCRIPT = DATA_DIR / 'build_qdrant_index.py'
QUERY_SCRIPT = DATA_DIR / 'query_qdrant.py'
SOURCES_FILE = DATA_DIR / 'sources.yaml'
REQ_FILE = DATA_DIR / 'requirements.txt'

print('ROOT      :', ROOT)
print('DATA_DIR  :', DATA_DIR)
print('BUILD     :', BUILD_SCRIPT.exists(), BUILD_SCRIPT)
print('QUERY     :', QUERY_SCRIPT.exists(), QUERY_SCRIPT)
print('SOURCES   :', SOURCES_FILE.exists(), SOURCES_FILE)
print('REQ       :', REQ_FILE.exists(), REQ_FILE)


## 1) Optional: Install Notebook Dependencies

Run this only if you have not installed dependencies yet in your current environment.


In [ ]:
RUN_INSTALL = False  # Set True if dependencies are not installed yet

if RUN_INSTALL:
    cmd = [sys.executable, '-m', 'pip', 'install', '-r', str(REQ_FILE)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))
else:
    print('Skipped. Set RUN_INSTALL=True to install dependencies.')


## 2) Configure Environment Variables

Required:
- `GOOGLE_API_KEY`
- `QDRANT_URL` (example: `http://localhost:6333`)

Optional:
- `QDRANT_API_KEY` (for cloud)
- `QDRANT_COLLECTION` (default: `star_learners_kb`)
- model overrides: `GEMINI_TEXT_EMBED_MODEL`, `GEMINI_IMAGE_EMBED_MODEL`, `GEMINI_CAPTION_MODEL`


In [ ]:
# Option A: set values directly in this notebook session (recommended for quick test)
# os.environ['GOOGLE_API_KEY'] = 'your_api_key'
# os.environ['QDRANT_URL'] = 'http://localhost:6333'
# os.environ['QDRANT_API_KEY'] = ''
# os.environ['QDRANT_COLLECTION'] = 'star_learners_kb'

required = ['GOOGLE_API_KEY', 'QDRANT_URL']
optional = ['QDRANT_API_KEY', 'QDRANT_COLLECTION', 'GEMINI_TEXT_EMBED_MODEL', 'GEMINI_IMAGE_EMBED_MODEL', 'GEMINI_CAPTION_MODEL']

print('Required checks:')
missing = []
for key in required:
    value = os.getenv(key)
    print(f'- {key}:', 'SET' if value else 'MISSING')
    if not value:
        missing.append(key)

print('
Optional checks:')
for key in optional:
    value = os.getenv(key)
    print(f'- {key}:', value if value else '(not set, default will be used)')

if missing:
    print('
Missing required env vars:', ', '.join(missing))
    print('Set them, then re-run this cell before ingestion.')
else:
    print('
Environment looks ready for ingestion.')


## 3) Inspect Source Inputs

This confirms exact website URLs and YouTube URL that will be indexed.


In [ ]:
import yaml

sources = yaml.safe_load(SOURCES_FILE.read_text(encoding='utf-8'))
print(json.dumps(sources, indent=2))


## 4) Run Ingestion

Choose one mode:
- `websites` for website-only indexing
- `youtube` for timestamped frame indexing
- `all` for both

Tips:
- Use `--recreate-collection` when you want a clean rebuild.
- Keep `--frame-interval-sec 5` for balanced precision vs cost.


In [ ]:
INGEST_MODE = 'all'              # 'websites' | 'youtube' | 'all'
FRAME_INTERVAL_SEC = 5
BATCH_SIZE = 32
RECREATE_COLLECTION = False

cmd = [
    sys.executable,
    str(BUILD_SCRIPT),
    '--mode', INGEST_MODE,
    '--frame-interval-sec', str(FRAME_INTERVAL_SEC),
    '--batch-size', str(BATCH_SIZE),
]

collection = os.getenv('QDRANT_COLLECTION')
if collection:
    cmd.extend(['--collection', collection])
if RECREATE_COLLECTION:
    cmd.append('--recreate-collection')

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))

print('
=== STDOUT ===')
print(result.stdout)
print('
=== STDERR ===')
print(result.stderr)
print('
Exit code:', result.returncode)

if result.returncode != 0:
    raise RuntimeError('Ingestion failed. Check logs above.')

# Parse JSON summary at the end for quick status
summary = None
for line in reversed(result.stdout.strip().splitlines()):
    line = line.strip()
    if line.startswith('{') and line.endswith('}'):
        try:
            summary = json.loads(line)
            break
        except Exception:
            pass

if summary is not None:
    print('
Parsed summary:')
    print(json.dumps(summary, indent=2))
else:
    print('
Could not parse structured summary JSON from stdout (see full output above).')


## 5) Query the Index

Try a tour/timestamp query first, then a curriculum/program query.


In [ ]:
def run_query(question: str, top_k: int = 5, source_type: str | None = None):
    cmd = [
        sys.executable,
        str(QUERY_SCRIPT),
        '--query', question,
        '--top-k', str(top_k),
    ]
    collection = os.getenv('QDRANT_COLLECTION')
    if collection:
        cmd.extend(['--collection', collection])
    if source_type in {'youtube', 'website'}:
        cmd.extend(['--source-type', source_type])

    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ROOT))

    if result.returncode != 0:
        print('STDOUT:
', result.stdout)
        print('STDERR:
', result.stderr)
        raise RuntimeError(f'Query failed with exit code {result.returncode}')

    payload = json.loads(result.stdout)
    print(json.dumps(payload, indent=2))
    return payload


tour_payload = run_query('show me the tour demo classroom', top_k=5)
program_payload = run_query('what is the infant care programme?', top_k=5)


## 6) Interpret Results

Use this checklist:
- For tour/demo queries, confirm at least one `youtube_frame` result with `timestamp_sec` and `youtube_deeplink`.
- For content/program queries, confirm relevant `website_chunk` hits from Star Learners URLs.
- If results are weak, rebuild with `RECREATE_COLLECTION=True` and run `INGEST_MODE='all'`.


In [ ]:
def summarize_hits(payload: dict):
    rows = payload.get('results', [])
    print('Query:', payload.get('query'))
    print('Total results:', len(rows))
    for i, row in enumerate(rows, start=1):
        print(f"{i}. {row.get('source_type')} | score={row.get('score'):.4f}")
        print('   url:', row.get('url'))
        if row.get('youtube_deeplink'):
            print('   timestamp:', row.get('timestamp_hms'), '|', row.get('youtube_deeplink'))
        print('   preview:', (row.get('content_preview') or '')[:140])

print('--- TOUR QUERY SUMMARY ---')
summarize_hits(tour_payload)
print('
--- PROGRAM QUERY SUMMARY ---')
summarize_hits(program_payload)


## Troubleshooting

- `Missing required environment variable`: set `GOOGLE_API_KEY` and `QDRANT_URL`.
- `ModuleNotFoundError`: run the install cell (`RUN_INSTALL=True`).
- Qdrant connection errors: verify Qdrant server is running and URL/API key are correct.
- Empty YouTube results: ensure ingestion ran with `INGEST_MODE='youtube'` or `all`.
- API/model errors: check quota, API key, and model override env vars.
